# InsightForge AI — Agent 2: Cleaning Agent
Imputes missing values (median for numeric, mode for categorical), drops duplicates, flags IQR outliers.


In [ ]:
%pip install pandas numpy
dbutils.library.restartPython()


In [ ]:
class InsightForgeState(TypedDict):
    """
    Shared state passed through all agents in the pipeline.
    Each agent reads what it needs and writes its output.
    No agent modifies another agent's output fields.

    Fields
    ------
    dataset_path     : path to the input CSV file
    gemini_key       : Gemini API key passed at runtime
    raw_df           : original DataFrame as uploaded
    cleaned_df       : DataFrame after cleaning agent runs
    schema_info      : column metadata detected by schema agent
    cleaning_report  : summary of all cleaning actions taken
    eda_results      : statistical analysis from EDA agent
    charts           : list of Plotly figure dicts from viz agent
    insights         : AI generated business insights text
    pdf_path         : path to the generated PDF report
    pipeline_log     : timestamped log of each agent execution
    errors           : list of error messages from any agent
    """
    dataset_path    : str
    gemini_key      : str
    raw_df          : Any
    cleaned_df      : Any
    schema_info     : dict
    cleaning_report : dict
    eda_results     : dict
    charts          : list
    insights        : str
    pdf_path        : str
    pipeline_log    : list
    errors          : list

print("✅ InsightForgeState defined")
print()
print("  State fields:")
fields = [
    ("dataset_path",     "input — path to CSV"),
    ("gemini_key",       "input — API key"),
    ("raw_df",           "Schema Agent reads this"),
    ("cleaned_df",       "Cleaning Agent writes this"),
    ("schema_info",      "Schema Agent writes this"),
    ("cleaning_report",  "Cleaning Agent writes this"),
    ("eda_results",      "EDA Agent writes this"),
    ("charts",           "Visualization Agent writes this"),
    ("insights",         "Insight Agent writes this"),
    ("pdf_path",         "Report Agent writes this"),
    ("pipeline_log",     "every agent appends to this"),
    ("errors",           "every agent appends on failure"),
]
for field, desc in fields:
    print(f"    {field:20} — {desc}")


In [ ]:
def log_event(state: InsightForgeState, agent: str, message: str) -> list:
    """
    Appends a timestamped log entry to the pipeline log.
    Called by every agent on start and completion.

    Parameters
    ----------
    state   : current pipeline state
    agent   : name of the calling agent
    message : what happened

    Returns
    -------
    list : updated pipeline log
    """
    timestamp = datetime.now().strftime("%H:%M:%S")
    entry     = f"[{timestamp}] {agent}: {message}"
    print(f"   {entry}")
    return state["pipeline_log"] + [entry]


def get_gemini_model() -> genai.GenerativeModel:
    """
    Returns a configured Gemini model instance.
    Re-reads the API key from widget each time to handle
    session restarts without needing to re-run setup cells.
    """
    key = dbutils.widgets.get("gemini_key")
    genai.configure(api_key=key)
    return genai.GenerativeModel(GEMINI_MODEL)


def safe_call_gemini(prompt: str, agent_name: str) -> str:
    """
    Wraps a Gemini API call with error handling.
    Cleans the response text to remove characters that
    fpdf2 cannot render with standard Helvetica font.

    Parameters
    ----------
    prompt     : the full prompt string to send
    agent_name : name of the calling agent for logging

    Returns
    -------
    str : cleaned response text or error message
    """
    try:
        m        = get_gemini_model()
        response = m.generate_content(prompt)
        text     = response.text

        # Remove characters unsupported by Helvetica in fpdf2
        replacements = {
            "\u2014": "-",    # em dash
            "\u2013": "-",    # en dash
            "\u2012": "-",    # figure dash
            "\u2011": "-",    # non-breaking hyphen
            "\u2010": "-",    # hyphen
            "\u2022": "-",    # bullet
            "\u2023": "-",    # triangle bullet
            "\u2043": "-",    # hyphen bullet
            "\u2018": "'",    # left single quote
            "\u2019": "'",    # right single quote
            "\u201a": "'",    # single low quote
            "\u201c": '"',    # left double quote
            "\u201d": '"',    # right double quote
            "\u201e": '"',    # double low quote
            "\u2026": "...",  # ellipsis
            "\u00a0": " ",    # non-breaking space
            "\u00b7": "-",    # middle dot
            "\u2015": "-",    # horizontal bar
        }
        for char, replacement in replacements.items():
            text = text.replace(char, replacement)

        # Final safety pass — replace remaining non-latin-1 chars
        text = text.encode("latin-1", errors="replace").decode("latin-1")
        return text

    except Exception as e:
        logger.warning(
            f"Gemini call failed in {agent_name}: {str(e)[:100]}"
        )
        print(f"   ⚠️  Gemini call failed in {agent_name}: {e}")
        return f"[Gemini error in {agent_name}: {str(e)}]"


print("✅ Utility functions defined")
print("   log_event()        — timestamped pipeline logging")
print("   get_gemini_model() — safe model initialisation")
print("   safe_call_gemini() — error handled API call with font cleaning")


In [ ]:
def cleaning_agent(state: InsightForgeState) -> dict:
    """
    Agent 2 — Cleaning Agent
    ------------------------
    Reads  : state["raw_df"]
    Writes : state["cleaned_df"], state["cleaning_report"]
    """
    agent_name = "Cleaning Agent"
    print(f"\n{'─' * 55}")
    print(f"🟡 {agent_name} starting...")

    df     = state["raw_df"].copy()
    errors = state["errors"]
    log    = log_event(state, agent_name, "started")

    logger.info(
        f"{agent_name} started — "
        f"{df.shape[0]} rows, {df.shape[1]} cols"
    )

    report = {
        "nulls_filled"       : {},
        "columns_dropped"    : [],
        "duplicates_removed" : 0,
        "outliers_flagged"   : {},
        "rows_before"        : int(len(df)),
        "rows_after"         : 0,
        "cols_before"        : int(df.shape[1]),
        "cols_after"         : 0,
    }

    try:
        for col in df.select_dtypes(include="number").columns:
            null_count = int(df[col].isnull().sum())
            if null_count > 0:
                fill_value = df[col].median()
                df[col]    = df[col].fillna(fill_value)
                report["nulls_filled"][col] = {
                    "strategy" : "median",
                    "value"    : round(float(fill_value), 4),
                    "count"    : null_count
                }
                print(
                    f"   Filled {null_count:4} nulls in "
                    f"'{col}' → median = {round(fill_value, 2)}"
                )

        for col in df.select_dtypes(include="object").columns:
            null_count  = int(df[col].isnull().sum())
            missing_pct = null_count / len(df) * 100
            if missing_pct > 70:
                df = df.drop(columns=[col])
                report["columns_dropped"].append(col)
                print(f"   Dropped '{col}' — {round(missing_pct, 1)}% missing")
                logger.warning(
                    f"{agent_name} — dropped '{col}' "
                    f"({round(missing_pct,1)}% missing)"
                )
            elif null_count > 0:
                fill_value = df[col].mode()[0]
                df[col]    = df[col].fillna(fill_value)
                report["nulls_filled"][col] = {
                    "strategy" : "mode",
                    "value"    : str(fill_value),
                    "count"    : null_count
                }
                print(
                    f"   Filled {null_count:4} nulls in "
                    f"'{col}' → mode = '{fill_value}'"
                )

        rows_before   = len(df)
        df            = df.drop_duplicates()
        dupes_removed = rows_before - len(df)
        report["duplicates_removed"] = dupes_removed

        if dupes_removed > 0:
            print(f"   Removed {dupes_removed} duplicate rows")
            logger.warning(
                f"{agent_name} — removed {dupes_removed} duplicates"
            )
        else:
            print(f"   No duplicate rows found")

        for col in df.select_dtypes(include="number").columns:
            q1  = df[col].quantile(0.25)
            q3  = df[col].quantile(0.75)
            iqr = q3 - q1
            lower = q1 - 1.5 * iqr
            upper = q3 + 1.5 * iqr
            outlier_count = int(
                ((df[col] < lower) | (df[col] > upper)).sum()
            )
            if outlier_count > 0:
                report["outliers_flagged"][col] = {
                    "count"       : outlier_count,
                    "lower_bound" : round(float(lower), 4),
                    "upper_bound" : round(float(upper), 4)
                }

        report["rows_after"] = int(len(df))
        report["cols_after"] = int(df.shape[1])

        log = log_event(
            state, agent_name,
            f"done — {len(report['nulls_filled'])} cols filled, "
            f"{len(report['columns_dropped'])} dropped, "
            f"{dupes_removed} dupes removed"
        )

        logger.info(
            f"{agent_name} complete — "
            f"{len(report['nulls_filled'])} cols filled, "
            f"{len(report['columns_dropped'])} dropped, "
            f"{dupes_removed} dupes removed"
        )

        print(f"   Rows : {report['rows_before']} → {report['rows_after']}")
        print(f"   Cols : {report['cols_before']} → {report['cols_after']}")
        print(f"✅ {agent_name} complete")

        return {
            "cleaned_df"     : df,
            "cleaning_report": report,
            "pipeline_log"   : log,
            "errors"         : errors
        }

    except Exception as e:
        msg = f"{agent_name} failed: {str(e)}"
        logger.error(f"{agent_name} FAILED — {str(e)}")
        print(f"   ❌ {msg}")
        return {
            "cleaned_df"     : state["raw_df"],
            "cleaning_report": report,
            "pipeline_log"   : log_event(
                state, agent_name, f"FAILED — {e}"
            ),
            "errors"         : errors + [msg]
        }

print("✅ cleaning_agent() defined")
